In [1]:
!pip install openai requests

In [12]:
import requests
import json
import re
from collections import Counter
from typing import Optional

In [ ]:
API_KEY ="YOUR_API_KEY"

In [32]:
MODEL = "meta-llama/llama-3-70b-instruct"



In [33]:
#function to call llm

def call_llm(prompt: str, model: str = MODEL, temperature: float = 0.7) -> Optional[str]:
    """Make API call to OpenRouter with error handling."""
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": 2000,
        },
        timeout=60,
    )

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None


In [34]:
print("TESTING API CONNECTION")

test_result = call_llm("Hello! Please respond with a short greeting.")
if test_result:
    print(f"✓ API Working!\nResponse: {test_result}")
else:
    print("✗ API failed. Check your API key and internet connection.")


TESTING API CONNECTION
✓ API Working!
Response: Hello! It's nice to meet you!


In [35]:
GSM8K_SAMPLES = [
    {
        "question": (
            "Carlos is planting a lemon tree. The tree will cost $90 to plant. "
            "Each year it will grow 7 lemons, which he can sell for $1.5 each. "
            "It costs $3 a year to water and feed the tree. How many years will "
            "it take before he starts earning money on the lemon tree?"
        ),
        "answer": "13",
    },
    {
        "question": (
            "Janet's ducks lay 16 eggs per day. She eats three for breakfast every "
            "morning and bakes muffins for her friends every day with four. She sells "
            "the remainder at the farmers' market daily for $2 per fresh duck egg. "
            "How much in dollars does she make every day at the farmers' market?"
        ),
        "answer": "18",
    },
    {
        "question": (
            "A robe takes 2 bolts of blue fiber and half that much white fiber. "
            "How many bolts in total does it take?"
        ),
        "answer": "3",
    },
]

In [36]:
#Core CRITIC-CoT functions


def generate_solution(question: str, model: str = MODEL) -> Optional[str]:
    """Generate a step-by-step solution with Chain-of-Thought."""
    prompt = f"""Solve this math problem step by step. Write each step as "Step X:" followed by the reasoning.
Your final answer must be in the format \\boxed{{answer}}.

Problem: {question}

Solution:"""
    return call_llm(prompt, model, temperature=0.7)


def critique_solution(question: str, solution: str, model: str = MODEL) -> Optional[str]:
    """Critique the solution step by step, stopping at the first error."""
    prompt = f"""How do you evaluate the following attempt with respect to the problem?

<problem>
{question}
</problem>

<attempt>
{solution}
</attempt>

**Notes**:
- Think step by step.
- At the end of evaluation for each step, state "Conclusion: Step [number] is correct" or "Conclusion: Step [number] is incorrect".
- Once a mistake is identified, stop the evaluation.
- If all steps are correct, state "All steps are correct."

Critique:"""
    return call_llm(prompt, model, temperature=0.5)


def refine_solution(
    question: str, solution: str, criticism: str, model: str = MODEL
) -> Optional[str]:
    """Refine the solution based on the critique."""
    prompt = f"""How do you refine the following attempt with respect to the problem, given the criticism?

<problem>
{question}
</problem>

<attempt>
{solution}
</attempt>

<criticism>
{criticism}
</criticism>

Provide a corrected solution. Start from the first error step if one was identified.
Your final answer must be in \\boxed{{answer}}.

Corrected solution:"""
    return call_llm(prompt, model, temperature=0.7)

In [45]:
#Answer extraction and normalisation
# ============================================
# ADD THESE FUNCTIONS AFTER critique_with_tools()
# ============================================

def refine_with_tools(question: str, solution: str, criticism: str, model: str = MODEL) -> Optional[str]:
    """
    Refine solution using tool-augmented critique.
    Adds verification results to the criticism for better refinement.
    """
    # Add verification results to prompt for better refinement
    verification = verify_with_code(solution)

    enhanced_criticism = criticism
    if verification["has_error"]:
        # Build detailed error summary
        error_summary = "\n\n## Arithmetic Verification Results:\n"
        for e in verification["errors"]:
            error_summary += f"- Step {e['step']}: {e['expression']} = {e['claimed']} "
            if 'actual' in e:
                error_summary += f"(correct value: {e['actual']})\n"
            else:
                error_summary += "(could not verify mathematically)\n"

        enhanced_criticism = criticism + error_summary

    return refine_solution(question, solution, enhanced_criticism, model)


def iterative_refine_with_tools(
    question: str, max_iterations: int = 3, verbose: bool = True
) -> Optional[str]:
    """
    Strategy A with tool-augmented critique.
    Uses Python executor to verify arithmetic steps.
    """
    if verbose:
        print(f"\n--- Iterative Refinement with Tools (max {max_iterations} rounds) ---")

    # Step 1: Generate initial solution
    current_solution = generate_solution(question)
    if not current_solution:
        return None

    # Step 2: Verify with tools first (arithmetic check)
    verification = verify_with_code(current_solution, verbose=verbose)

    # Step 3: If no arithmetic errors, check logic with LLM
    if not verification["has_error"]:
        first_critique = critique_solution(question, current_solution)
        if first_critique and not _has_error(first_critique):
            if verbose:
                print("  ✓ Initial solution already correct (arithmetic + logic verified).")
            return current_solution
        elif verbose and verification["verified"]:
            print(f"  ✓ Arithmetic verified ({len(verification['verified'])} steps correct)")
            print("  ⚠ Checking logic errors...")
    elif verbose:
        print(f"  ✗ Found {len(verification['errors'])} arithmetic error(s)")

    # Step 4: Refinement loop
    for i in range(max_iterations):
        if verbose:
            print(f"\n  Round {i + 1}:")

        # Use tool-augmented critique
        critique = critique_with_tools(question, current_solution)
        if not critique:
            if verbose:
                print("    Critique failed, stopping.")
            break

        if not _has_error(critique):
            if verbose:
                print("    ✓ No errors found. Solution accepted.")
            break

        if verbose:
            print("    ✗ Errors found. Refining with tool feedback...")

        refined = refine_with_tools(question, current_solution, critique)
        if not refined:
            if verbose:
                print("    Refinement failed.")
            break

        current_solution = refined

    return current_solution


# ============================================
# OPTIONAL: Add a combined strategy that uses tools for filtering
# ============================================

def critic_as_filter_with_tools(
    question: str, num_samples: int = 3, verbose: bool = True
) -> Optional[str]:
    """
    Strategy B with tool-augmented filtering.
    Generates multiple solutions and uses arithmetic verification to filter.
    """
    if verbose:
        print(f"\n--- Critic as Filter with Tools (generating {num_samples} solutions) ---")

    solutions = []
    for i in range(num_samples):
        if verbose:
            print(f"  Generating solution {i + 1}/{num_samples}...")
        sol = generate_solution(question)
        if sol:
            solutions.append(sol)

    if not solutions:
        return None

    # Filter using arithmetic verification
    valid_solutions = []
    if verbose:
        print("  Filtering solutions with arithmetic verification...")

    for idx, sol in enumerate(solutions):
        verification = verify_with_code(sol)

        if not verification["has_error"]:
            # No arithmetic errors, also check logic with LLM
            critique = critique_solution(question, sol)
            if critique and not _has_error(critique):
                valid_solutions.append(sol)
                if verbose:
                    print(f"    ✓ Solution {idx + 1} passed verification")
            else:
                if verbose:
                    print(f"    ⚠ Solution {idx + 1} has logic errors (rejected)")
        else:
            if verbose:
                print(f"    ✗ Solution {idx + 1} rejected: {len(verification['errors'])} arithmetic error(s)")

    if valid_solutions:
        if verbose:
            print(f"  {len(valid_solutions)} valid solution(s) found.")
        return valid_solutions[0]

    if verbose:
        print("  No valid solutions; returning first candidate as fallback.")
    return solutions[0] if solutions else None

In [46]:
print("TESTING CORE FUNCTIONS")


question = GSM8K_SAMPLES[0]["question"]
print(f"Question: {question[:100]}...\n")

print(" Generating Solution ")
solution = generate_solution(question)
if solution:
    print(f"Solution (first 500 chars):\n{solution[:500]}...\n")
else:
    print("Failed to generate solution")

print(" Critiquing Solution ")
critique = critique_solution(question, solution or "")
if critique:
    print(f"Critique (first 500 chars):\n{critique[:500]}...\n")
else:
    print("Failed to critique")

print(" Refining Solution ")
refined = refine_solution(question, solution or "", critique or "")
if refined:
    print(f"Refined (first 500 chars):\n{refined[:500]}...\n")
    raw_answer = extract_answer(refined)
    print(f"Extracted Answer : {raw_answer}")
    print(f"Normalised Answer: {normalize_answer(raw_answer)}")
    print(f"Expected Answer  : {GSM8K_SAMPLES[0]['answer']}")
    print(f"Match            : {answers_match(raw_answer, GSM8K_SAMPLES[0]['answer'])}")
else:
    print("Failed to refine")

TESTING CORE FUNCTIONS
Question: Carlos is planting a lemon tree. The tree will cost $90 to plant. Each year it will grow 7 lemons, w...

 Generating Solution 
Solution (first 500 chars):
Here is the step-by-step solution:

Step 1: Calculate the total revenue from selling lemons per year.
The tree grows 7 lemons per year, and each lemon can be sold for $1.5, so the total revenue per year is:
7 lemons/year × $1.5/lemon = $10.50/year

Step 2: Calculate the total cost per year.
The cost to water and feed the tree per year is $3. So, the total cost per year is:
$3/year

Step 3: Calculate the net profit per year.
The net profit per year is the revenue minus the cost:
$10.50/year - $3/...

 Critiquing Solution 
Critique (first 500 chars):
Here is the evaluation of the attempt:

**Step 1: Calculate the total revenue from selling lemons per year.**
The calculation is correct: 7 lemons/year × $1.5/lemon = $10.50/year
Conclusion: Step 1 is correct.

**Step 2: Calculate the total cost per year.**

In [51]:
#Inference strategies


def _has_error(critique: str) -> bool:

    """
    Check if critique contains any error indicators.
    Handles both LLM critique and tool-augmented critique.
    """
    if not critique:
        return False

    text = critique.lower()

    # Check for numbered step errors (from LLM)
    has_numbered_error = bool(re.search(r"step \d+ is incorrect", text))

    # Check for error keywords
    has_keywords = any(kw in text for kw in ("incorrect", "wrong", "mistake", "error"))

    # Check for tool-augmented error indicators
    has_tool_error = any(kw in text for kw in ("arithmetic error", "should be", "could not verify"))

    return has_numbered_error or has_keywords or has_tool_error







def iterative_refine(
    question: str, max_iterations: int = 3, verbose: bool = True
) -> Optional[str]:
    """Strategy A: Iteratively refine until no errors are found."""
    if verbose:
        print(f"\n--- Iterative Refinement (max {max_iterations} rounds) ---")

    # Step 1: Generate initial solution
    current_solution = generate_solution(question)
    if not current_solution:
        return None

    # Step 2: CRITICAL - Check if initial solution is already correct
    # This must be BEFORE the loop
    if verbose:
        print("  Checking initial solution...")

    first_critique = critique_solution(question, current_solution)
    if first_critique and not _has_error(first_critique):
        if verbose:
            print("  ✓ Initial solution already correct, skipping refinement.")
        return current_solution

    # Step 3: Only enter refinement loop if errors were found
    for i in range(max_iterations):
        if verbose:
            print(f"\n  Round {i + 1}:")

        critique = critique_solution(question, current_solution)
        if not critique:
            if verbose:
                print("    Critique failed, stopping.")
            break

        if not _has_error(critique):
            if verbose:
                print("    ✓ No errors found. Solution accepted.")
            break

        if verbose:
            print("    ✗ Errors found. Refining...")

        refined = refine_solution(question, current_solution, critique)
        if not refined:
            if verbose:
                print("    Refinement failed.")
            break

        current_solution = refined

    return current_solution


def critic_as_filter(
    question: str, num_samples: int = 3, verbose: bool = True
) -> Optional[str]:
    """Strategy B: Generate multiple solutions, keep only critic-approved ones."""
    if verbose:
        print(f"\n--- Critic as Filter (generating {num_samples} solutions) ---")

    solutions = []
    for i in range(num_samples):
        if verbose:
            print(f"  Generating solution {i + 1}/{num_samples}...")
        sol = generate_solution(question)
        if sol:
            solutions.append(sol)

    if not solutions:
        return None

    valid = []
    if verbose:
        print("  Filtering solutions...")

    for idx, sol in enumerate(solutions):
        critique = critique_solution(question, sol)
        if critique is None:
            continue
        # BUG FIX: use _has_error() so "wrong" / "mistake" are also caught
        if not _has_error(critique):
            valid.append(sol)
            if verbose:
                print(f"    ✓ Solution {idx + 1} passed the critic.")
        else:
            if verbose:
                print(f"    ✗ Solution {idx + 1} rejected by critic.")

    if valid:
        if verbose:
            print(f"  {len(valid)} valid solution(s) found.")
        return valid[0]

    if verbose:
        print("  No valid solutions; returning first candidate as fallback.")
    return solutions[0]


def majority_vote(
    question: str, num_samples: int = 5, verbose: bool = True
) -> Optional[str]:
    """
    Strategy C (NEW): Sample k solutions and return the one whose extracted
    answer appears most often (self-consistency / majority vote).
    This is a standard CRITIC-CoT baseline absent from the original code.
    """
    if verbose:
        print(f"\n--- Majority Vote (sampling {num_samples} solutions) ---")

    solutions = []
    answers = []
    for i in range(num_samples):
        if verbose:
            print(f"  Sample {i + 1}/{num_samples}...")
        sol = generate_solution(question)
        if sol:
            ans = normalize_answer(extract_answer(sol))
            solutions.append(sol)
            answers.append(ans)
            if verbose:
                print(f"    → answer: {ans}")

    if not answers:
        return None

    counts = Counter(a for a in answers if a is not None)
    if not counts:
        return solutions[0] if solutions else None

    best_answer = counts.most_common(1)[0][0]
    if verbose:
        print(f"  Vote tally: {dict(counts)}")
        print(f"  Winner: {best_answer}")

    # Return the solution whose extracted answer matches the winner
    for sol, ans in zip(solutions, answers):
        if ans == best_answer:
            return sol
    return solutions[0]

In [52]:
#Smoke-test all three strategies

print("TESTING INFERENCE STRATEGIES")


question = GSM8K_SAMPLES[0]["question"]

for label, strategy_fn, kwargs in [
    ("A: Iterative Refinement", iterative_refine, {"max_iterations": 2}),
    ("B: Critic as Filter",     critic_as_filter, {"num_samples": 3}),
    ("C: Majority Vote",        majority_vote,    {"num_samples": 5}),
]:
    print("\n" + "=" * 40)
    print(f"STRATEGY {label}")
    print("=" * 40)
    result = strategy_fn(question, **kwargs)
    if result:
        raw = extract_answer(result)
        print(f"\nFinal Answer : {normalize_answer(raw)}")
        print(f"Expected     : {GSM8K_SAMPLES[0]['answer']}")
        print(f"Match        : {answers_match(raw, GSM8K_SAMPLES[0]['answer'])}")
    else:
        print("Strategy returned no result.")



TESTING INFERENCE STRATEGIES

STRATEGY A: Iterative Refinement

--- Iterative Refinement (max 2 rounds) ---
  Checking initial solution...
  ✓ Initial solution already correct, skipping refinement.

Final Answer : 12
Expected     : 13
Match        : False

STRATEGY B: Critic as Filter

--- Critic as Filter (generating 3 solutions) ---
  Generating solution 1/3...
  Generating solution 2/3...
  Generating solution 3/3...
  Filtering solutions...
    ✓ Solution 1 passed the critic.
    ✓ Solution 2 passed the critic.
    ✗ Solution 3 rejected by critic.
  2 valid solution(s) found.

Final Answer : 13
Expected     : 13
Match        : True

STRATEGY C: Majority Vote

--- Majority Vote (sampling 5 solutions) ---
  Sample 1/5...
    → answer: 12
  Sample 2/5...
    → answer: 13
  Sample 3/5...
    → answer: 13
  Sample 4/5...
    → answer: 13
  Sample 5/5...
    → answer: 13
  Vote tally: {'12': 1, '13': 4}
  Winner: 13

Final Answer : 13
Expected     : 13
Match        : True


In [53]:
#Side-by-side evaluation

print("SIDE-BY-SIDE EVALUATION")
print("=" * 50)

strategies = {
    "Baseline":   lambda q: generate_solution(q),
    "Iter-refine": lambda q: iterative_refine(q, max_iterations=2, verbose=False),
    "Filter":      lambda q: critic_as_filter(q, num_samples=3, verbose=False),
    "Majority":    lambda q: majority_vote(q, num_samples=5, verbose=False),
}

# Track per-strategy accuracy
totals = {name: 0 for name in strategies}
correct = {name: 0 for name in strategies}

for i, sample in enumerate(GSM8K_SAMPLES):
    print(f"\n--- Sample {i + 1} ---")
    print(f"Question: {sample['question'][:80]}...")
    print(f"Expected: {sample['answer']}")

    for name, fn in strategies.items():
        sol = fn(sample["question"])
        raw = extract_answer(sol) if sol else None
        match = answers_match(raw, sample["answer"])
        totals[name] += 1
        if match:
            correct[name] += 1
        mark = "correct" if match else "wrong"
        print(f"  {name:<15} pred={normalize_answer(raw):<8} {mark}")


print("ACCURACY SUMMARY")

for name in strategies:
    acc = correct[name] / totals[name] * 100 if totals[name] else 0
    print(f"  {name:<15} {correct[name]}/{totals[name]}  ({acc:.0f}%)")


SIDE-BY-SIDE EVALUATION

--- Sample 1 ---
Question: Carlos is planting a lemon tree. The tree will cost $90 to plant. Each year it w...
Expected: 13
  Baseline        pred=12       wrong
  Iter-refine     pred=12       wrong
  Filter          pred=13       correct
  Majority        pred=13       correct

--- Sample 2 ---
Question: Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning an...
Expected: 18
  Baseline        pred=18       correct
  Iter-refine     pred=18       correct
  Filter          pred=18       correct
  Majority        pred=18       correct

--- Sample 3 ---
Question: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolt...
Expected: 3
  Baseline        pred=3        correct
  Iter-refine     pred=3        correct
  Filter          pred=3        correct
  Majority        pred=3        correct
ACCURACY SUMMARY
  Baseline        2/3  (67%)
  Iter-refine     2/3  (67%)
  Filter          3/3  (100%)
  Majority        3/3